In [ ]:
import kagglehub
path = kagglehub.dataset_download("rtatman/glove-global-vectors-for-word-representation")
print("Path to dataset files:", path)


In [3]:
# ============================================================
# Kaggle: Add this dataset to your notebook before running:
# "glove-global-vectors-for-word-representation"
# Datasets → Add Data → search "glove 6b" → add it
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import numpy as np
import random
import json
import os



# ============================================================
# Reproducibility
# ============================================================
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# Config
# ============================================================
EMBEDDING_DIM = 100
HIDDEN_DIM    = 128
N_LAYERS      = 1
DROPOUT       = 0.3
BATCH_SIZE    = 128
N_EPOCHS      = 20
LR            = 1e-3
MAX_VOCAB     = 10000
MAX_SEQ_LEN   = 30
PATIENCE      = 3
path = "/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation"
GLOVE_PATH    = f"{path}/glove.6B.100d.txt" 
SAVE_DIR      = "/kaggle/working"
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================
# 1. Load Data
# ============================================================
print("\nLoading SST-2...")
dataset    = load_dataset("glue", "sst2")
train_data = dataset["train"]
val_data   = dataset["validation"]

# ============================================================
# 2. Vocabulary
# ============================================================
def tokenize(text):
    return text.lower().split()

counter = Counter()
for item in train_data:
    counter.update(tokenize(item["sentence"]))

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

def encode(sentence):
    tokens = tokenize(sentence)[:MAX_SEQ_LEN]
    return [vocab.get(t, 1) for t in tokens]

print(f"Vocabulary size: {len(vocab)}")

# ============================================================
# 3. Load GloVe
# ============================================================
def load_glove(glove_path, vocab, embedding_dim):
    print(f"\nLoading GloVe from {glove_path}...")
    glove = {}
    with open(glove_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            word  = parts[0]
            vec   = np.array(parts[1:], dtype=np.float32)
            glove[word] = vec

    matrix = np.zeros((len(vocab), embedding_dim), dtype=np.float32)
    found  = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
            found += 1
        else:
            matrix[idx] = np.random.normal(0, 0.1, embedding_dim)

    print(f"GloVe coverage: {found}/{len(vocab)} words ({100*found/len(vocab):.1f}%)")
    return torch.tensor(matrix)

glove_matrix = load_glove(GLOVE_PATH, vocab, EMBEDDING_DIM)

# ============================================================
# 4. Dataset
# ============================================================
class SentimentDataset(Dataset):
    def __init__(self, data):
        self.samples = [(encode(item["sentence"]), item["label"]) for item in data]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    seqs, labels = zip(*batch)
    max_len = max(len(s) for s in seqs)
    padded  = [s + [0] * (max_len - len(s)) for s in seqs]
    return (torch.tensor(padded, dtype=torch.long),
            torch.tensor(labels, dtype=torch.long))

train_dataset = SentimentDataset(train_data)
val_dataset   = SentimentDataset(val_data)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# ============================================================
# 5. LSTM Building Blocks
# ============================================================
def lstm_cell(x_t, h_t, c_t, params):
    i_t = torch.sigmoid(x_t @ params["W_ii"].T + h_t @ params["W_hi"].T + params["b_i"])
    f_t = torch.sigmoid(x_t @ params["W_if"].T + h_t @ params["W_hf"].T + params["b_f"])
    o_t = torch.sigmoid(x_t @ params["W_io"].T + h_t @ params["W_ho"].T + params["b_o"])
    g_t = torch.tanh(   x_t @ params["W_ig"].T + h_t @ params["W_hg"].T + params["b_g"])
    c_t = f_t * c_t + i_t * g_t
    h_t = o_t * torch.tanh(c_t)
    return h_t, c_t

def make_lstm_layer(input_size, hidden_size):
    return nn.ParameterDict({
        "W_ii": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hi": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_i":  nn.Parameter(torch.zeros(hidden_size)),

        "W_if": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hf": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_f":  nn.Parameter(torch.zeros(hidden_size)),

        "W_io": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_ho": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_o":  nn.Parameter(torch.zeros(hidden_size)),

        "W_ig": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hg": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_g":  nn.Parameter(torch.zeros(hidden_size)),
    })

# ============================================================
# 6. Experiment 1: Standard BiLSTM (Separate Weights)
# ============================================================
class StandardBiLSTM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, n_layers, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_dim
        self.num_layers  = n_layers
        self.fwd_layers  = nn.ModuleList()
        self.bwd_layers  = nn.ModuleList()

        for layer in range(n_layers):
            in_size = embedding_dim if layer == 0 else hidden_dim
            self.fwd_layers.append(make_lstm_layer(in_size, hidden_dim))
            self.bwd_layers.append(make_lstm_layer(in_size, hidden_dim))

        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, x, state=None):
        seq_len, batch, _ = x.size()

        if state is None:
            h_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            h_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        else:
            (h_f, c_f), (h_b, c_b) = state

        layer_input = x

        for layer_idx in range(self.num_layers):
            fwd_params = self.fwd_layers[layer_idx]
            bwd_params = self.bwd_layers[layer_idx]

            h_t_f = h_f[layer_idx].clone()
            c_t_f = c_f[layer_idx].clone()
            h_t_b = h_b[layer_idx].clone()
            c_t_b = c_b[layer_idx].clone()

            fwd_outputs = []
            for t in range(seq_len):
                h_t_f, c_t_f = lstm_cell(layer_input[t], h_t_f, c_t_f, fwd_params)
                fwd_outputs.append(h_t_f.unsqueeze(0))

            bwd_outputs = [None] * seq_len
            for t in reversed(range(seq_len)):
                h_t_b, c_t_b = lstm_cell(layer_input[t], h_t_b, c_t_b, bwd_params)
                bwd_outputs[t] = h_t_b.unsqueeze(0)

            fwd = torch.cat(fwd_outputs, dim=0)
            bwd = torch.cat(bwd_outputs, dim=0)
            layer_output = fwd + bwd

            h_f[layer_idx] = h_t_f
            c_f[layer_idx] = c_t_f
            h_b[layer_idx] = h_t_b
            c_b[layer_idx] = c_t_b

            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)

            layer_input = layer_output

        return layer_input, ((h_f, c_f), (h_b, c_b))

# ============================================================
# 7. Experiment 2: Reversible LSTM (Shared Weights)
# ============================================================
class ReversibleLSTM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, n_layers, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_dim
        self.num_layers  = n_layers
        self.layers      = nn.ModuleList()

        for layer in range(n_layers):
            in_size = embedding_dim if layer == 0 else hidden_dim
            self.layers.append(make_lstm_layer(in_size, hidden_dim))

        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, x, state=None):
        seq_len, batch, _ = x.size()

        if state is None:
            h_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            h_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        else:
            (h_f, c_f), (h_b, c_b) = state

        layer_input = x

        for layer_idx, params in enumerate(self.layers):

            h_t_f = h_f[layer_idx].clone()
            c_t_f = c_f[layer_idx].clone()
            h_t_b = h_b[layer_idx].clone()
            c_t_b = c_b[layer_idx].clone()

            fwd_outputs = []
            for t in range(seq_len):
                h_t_f, c_t_f = lstm_cell(layer_input[t], h_t_f, c_t_f, params)
                fwd_outputs.append(h_t_f.unsqueeze(0))

            bwd_outputs = [None] * seq_len
            for t in reversed(range(seq_len)):
                h_t_b, c_t_b = lstm_cell(layer_input[t], h_t_b, c_t_b, params)
                bwd_outputs[t] = h_t_b.unsqueeze(0)

            fwd = torch.cat(fwd_outputs, dim=0)
            bwd = torch.cat(bwd_outputs, dim=0)
            layer_output = fwd + bwd

            h_f[layer_idx] = h_t_f
            c_f[layer_idx] = c_t_f
            h_b[layer_idx] = h_t_b
            c_b[layer_idx] = c_t_b

            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)

            layer_input = layer_output

        return layer_input, ((h_f, c_f), (h_b, c_b))

# ============================================================
# 8. Classifier Wrapper
# ============================================================
class SentimentClassifier(nn.Module):
    def __init__(self, lstm_model, vocab_size, embedding_dim, hidden_dim, glove_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(glove_matrix)
        self.embedding.weight.requires_grad = False   # frozen GloVe

        self.lstm    = lstm_model
        self.dropout = nn.Dropout(DROPOUT)
        self.fc      = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        embedded      = self.embedding(x)
        embedded      = embedded.permute(1, 0, 2)
        out, ((h_f, _), (h_b, _)) = self.lstm(embedded)
        sentence_repr = self.dropout(h_f[-1] + h_b[-1])
        return self.fc(sentence_repr)

# ============================================================
# 9. Parameter Counting
# ============================================================
def count_params(model, lstm_only=False):
    if lstm_only:
        return sum(p.numel() for p in model.lstm.parameters())
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ============================================================
# 10. Training and Evaluation
# ============================================================
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for seqs, labels in loader:
        seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(seqs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for seqs, labels in loader:
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            logits = model(seqs)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total

def run_training(model, name):
    model     = model.to(DEVICE)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR
    )
    criterion = nn.CrossEntropyLoss()

    total_params = count_params(model, lstm_only=False)
    lstm_params  = count_params(model, lstm_only=True)

    print(f"\n{'='*60}")
    print(f"Training  : {name}")
    print(f"Total trainable params : {total_params:,}")
    print(f"LSTM-only params       : {lstm_params:,}")
    print(f"{'='*60}")

    best_val_acc     = 0.0
    best_epoch       = 0
    patience_counter = 0
    best_state       = None
    history          = []

    for epoch in range(N_EPOCHS):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)

        improved = " ✓" if vl_acc > best_val_acc else ""
        print(f"Epoch {epoch+1:>2}/{N_EPOCHS} | "
              f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
              f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}{improved}")

        history.append({
            "epoch": epoch + 1,
            "train_loss": tr_loss, "train_acc": tr_acc,
            "val_loss":   vl_loss, "val_acc":   vl_acc,
        })

        if vl_acc > best_val_acc:
            best_val_acc     = vl_acc
            best_epoch       = epoch + 1
            patience_counter = 0
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}. "
                      f"Best val acc: {best_val_acc:.4f} at epoch {best_epoch}")
                break

    model.load_state_dict(best_state)
    print(f"Restored best model from epoch {best_epoch} "
          f"(val acc: {best_val_acc:.4f})")
    return model, best_val_acc, history

# ============================================================
# 11. Build and Train Both Models
# ============================================================
sep_lstm    = StandardBiLSTM(EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
shared_lstm = ReversibleLSTM(EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)

sep_classifier    = SentimentClassifier(sep_lstm,    len(vocab), EMBEDDING_DIM, HIDDEN_DIM, glove_matrix.clone())
shared_classifier = SentimentClassifier(shared_lstm, len(vocab), EMBEDDING_DIM, HIDDEN_DIM, glove_matrix.clone())

sep_classifier,    sep_best,    sep_history    = run_training(sep_classifier,    "Experiment 1: Separate Weights")
shared_classifier, shared_best, shared_history = run_training(shared_classifier, "Experiment 2: Shared Weights")

# ============================================================
# 12. Post-Training Representation Analysis
# ============================================================
test_sentences = {
    "S1: terrible_start good_end"  : "terrible movie great ending",
    "S2: terrible_start bad_end"   : "terrible movie terrible ending",
    "S3: great_start good_end"     : "great movie great ending",
    "S4: great_start bad_end"      : "great movie terrible ending",
}

def get_representations(classifier, sentence):
    classifier.eval()
    tokens = encode(sentence)
    x      = torch.tensor(tokens).unsqueeze(0).to(DEVICE)
    emb    = classifier.embedding(x).permute(1, 0, 2)
    with torch.no_grad():
        _, ((h_f, _), (h_b, _)) = classifier.lstm(emb)
    return h_f[-1].squeeze(0), h_b[-1].squeeze(0)

def cos_sim(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

def analyze(classifier, name):
    print(f"\n{'='*60}")
    print(f"Representation Analysis: {name}")
    print(f"{'='*60}")

    reprs = {}
    for label, sent in test_sentences.items():
        fwd, bwd = get_representations(classifier, sent)
        reprs[label] = {"fwd": fwd, "bwd": bwd, "combined": fwd + bwd}

    labels = list(reprs.keys())
    analysis = {}

    print("\n--- Forward Pass (should group by ENDING) ---")
    s1s3 = cos_sim(reprs[labels[0]]['fwd'], reprs[labels[2]]['fwd'])
    s2s4 = cos_sim(reprs[labels[1]]['fwd'], reprs[labels[3]]['fwd'])
    s1s2 = cos_sim(reprs[labels[0]]['fwd'], reprs[labels[1]]['fwd'])
    print(f"S1 vs S3 (both great ending)    : {s1s3:.4f}")
    print(f"S2 vs S4 (both terrible ending) : {s2s4:.4f}")
    print(f"S1 vs S2 (different ending)     : {s1s2:.4f}")
    analysis["fwd"] = {"s1_s3": s1s3, "s2_s4": s2s4, "s1_s2": s1s2}

    print("\n--- Backward Pass (should group by START) ---")
    b_s1s2 = cos_sim(reprs[labels[0]]['bwd'], reprs[labels[1]]['bwd'])
    b_s3s4 = cos_sim(reprs[labels[2]]['bwd'], reprs[labels[3]]['bwd'])
    b_s1s3 = cos_sim(reprs[labels[0]]['bwd'], reprs[labels[2]]['bwd'])
    print(f"S1 vs S2 (both terrible start)  : {b_s1s2:.4f}")
    print(f"S3 vs S4 (both great start)     : {b_s3s4:.4f}")
    print(f"S1 vs S3 (different start)      : {b_s1s3:.4f}")
    analysis["bwd"] = {"s1_s2": b_s1s2, "s3_s4": b_s3s4, "s1_s3": b_s1s3}

    print("\n--- Combined (all four should be distinct) ---")
    combined = {}
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if j > i:
                sim = cos_sim(reprs[l1]["combined"], reprs[l2]["combined"])
                key = f"{l1[:2]}_vs_{l2[:2]}"
                combined[key] = sim
                print(f"  {l1[:2]} vs {l2[:2]}: {sim:.4f}")
    analysis["combined"] = combined

    return analysis

sep_analysis    = analyze(sep_classifier,    "Experiment 1: Separate Weights")
shared_analysis = analyze(shared_classifier, "Experiment 2: Shared Weights")

# ============================================================
# 13. Final Summary
# ============================================================
sep_lstm_params    = count_params(sep_classifier,    lstm_only=True)
shared_lstm_params = count_params(shared_classifier, lstm_only=True)
lstm_reduction     = (1 - shared_lstm_params / sep_lstm_params) * 100
sep_total          = count_params(sep_classifier,    lstm_only=False)
shared_total       = count_params(shared_classifier, lstm_only=False)
acc_drop           = (sep_best - shared_best) * 100

print(f"\n{'='*60}")
print(f"Final Summary")
print(f"{'='*60}")
print(f"{'Metric':<35} {'Separate':>10} {'Shared':>10}")
print(f"{'-'*55}")
print(f"{'Best Val Accuracy':<35} {sep_best:>10.4f} {shared_best:>10.4f}")
print(f"{'Total Trainable Params':<35} {sep_total:>10,} {shared_total:>10,}")
print(f"{'LSTM-only Params':<35} {sep_lstm_params:>10,} {shared_lstm_params:>10,}")
print(f"{'LSTM Param Reduction':<35} {'baseline':>10} {lstm_reduction:>9.1f}%")
print(f"{'Accuracy Drop':<35} {'baseline':>10} {acc_drop:>9.2f}%")

# ============================================================
# 14. Save Everything to /kaggle/working
# ============================================================
results = {
    "config": {
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim":    HIDDEN_DIM,
        "n_layers":      N_LAYERS,
        "dropout":       DROPOUT,
        "batch_size":    BATCH_SIZE,
        "lr":            LR,
        "patience":      PATIENCE,
    },
    "experiment_1_separate": {
        "best_val_acc":  sep_best,
        "lstm_params":   sep_lstm_params,
        "total_params":  sep_total,
        "history":       sep_history,
        "analysis":      sep_analysis,
    },
    "experiment_2_shared": {
        "best_val_acc":  shared_best,
        "lstm_params":   shared_lstm_params,
        "total_params":  shared_total,
        "history":       shared_history,
        "analysis":      shared_analysis,
    },
    "comparison": {
        "lstm_reduction_pct": lstm_reduction,
        "accuracy_drop_pct":  acc_drop,
    }
}

with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)

torch.save(sep_classifier.state_dict(),    f"{SAVE_DIR}/sep_classifier.pt")
torch.save(shared_classifier.state_dict(), f"{SAVE_DIR}/shared_classifier.pt")

print(f"\nSaved to {SAVE_DIR}:")
print(f"  results.json          — all metrics and analysis")
print(f"  sep_classifier.pt     — Experiment 1 best weights")
print(f"  shared_classifier.pt  — Experiment 2 best weights")

Device: cuda

Loading SST-2...
Vocabulary size: 10000

Loading GloVe from /kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt...
GloVe coverage: 9487/10000 words (94.9%)
Train: 67349 | Val: 872

Training  : Experiment 1: Separate Weights
Total trainable params : 234,754
LSTM-only params       : 234,496
Epoch  1/20 | Train Loss: 0.4360 Acc: 0.7901 | Val Loss: 0.4205 Acc: 0.8028 ✓
Epoch  2/20 | Train Loss: 0.3440 Acc: 0.8454 | Val Loss: 0.4437 Acc: 0.8142 ✓
Epoch  3/20 | Train Loss: 0.2957 Acc: 0.8717 | Val Loss: 0.4641 Acc: 0.8039
Epoch  4/20 | Train Loss: 0.2527 Acc: 0.8927 | Val Loss: 0.4492 Acc: 0.8200 ✓
Epoch  5/20 | Train Loss: 0.2210 Acc: 0.9078 | Val Loss: 0.5042 Acc: 0.8096
Epoch  6/20 | Train Loss: 0.1990 Acc: 0.9184 | Val Loss: 0.5548 Acc: 0.8062
Epoch  7/20 | Train Loss: 0.1790 Acc: 0.9274 | Val Loss: 0.5843 Acc: 0.8073

Early stopping at epoch 7. Best val acc: 0.8200 at epoch 4
Restored best model from epoch 4 (val acc: 0.8200)

Trai

In [1]:
import torch
import torch.nn as nn

# ============================================================
# Explicit Weight Sharing BiLSTM
# ============================================================
class SharedWeightBiLSTM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, n_layers, dropout=0.0):
        super().__init__()
        self.hidden_size   = hidden_dim
        self.num_layers    = n_layers
        self.fwd_layers    = nn.ModuleList()
        self.bwd_layers    = nn.ModuleList()
        self.dropout_layer = nn.Dropout(dropout)

        for layer in range(n_layers):
            in_size = embedding_dim if layer == 0 else hidden_dim

            # Step 1: create forward layer — fresh parameters
            fwd_layer = self._make_lstm_layer(in_size, hidden_dim)

            # Step 2: backward layer explicitly shares forward parameters
            bwd_layer = self._make_lstm_layer(in_size, hidden_dim,
                                              shared_layer=fwd_layer)

            self.fwd_layers.append(fwd_layer)
            self.bwd_layers.append(bwd_layer)

    def _make_lstm_layer(self, input_size, hidden_size, shared_layer=None):
        """
        Create parameters for one LSTM layer.
        If shared_layer is provided, return it directly —
        explicit architectural weight sharing.
        """
        if shared_layer is not None:
            return shared_layer          # same object, same memory

        return nn.ParameterDict({
            "W_ii": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
            "W_hi": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
            "b_i":  nn.Parameter(torch.zeros(hidden_size)),

            "W_if": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
            "W_hf": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
            "b_f":  nn.Parameter(torch.zeros(hidden_size)),

            "W_io": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
            "W_ho": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
            "b_o":  nn.Parameter(torch.zeros(hidden_size)),

            "W_ig": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
            "W_hg": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
            "b_g":  nn.Parameter(torch.zeros(hidden_size)),
        })

    def _lstm_cell(self, x_t, h_t, c_t, params):
        i_t = torch.sigmoid(x_t @ params["W_ii"].T + h_t @ params["W_hi"].T + params["b_i"])
        f_t = torch.sigmoid(x_t @ params["W_if"].T + h_t @ params["W_hf"].T + params["b_f"])
        o_t = torch.sigmoid(x_t @ params["W_io"].T + h_t @ params["W_ho"].T + params["b_o"])
        g_t = torch.tanh(   x_t @ params["W_ig"].T + h_t @ params["W_hg"].T + params["b_g"])
        c_t = f_t * c_t + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        return h_t, c_t

    def forward(self, x, state=None):
        seq_len, batch, _ = x.size()

        if state is None:
            h_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            h_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        else:
            (h_f, c_f), (h_b, c_b) = state

        layer_input = x

        for layer_idx in range(self.num_layers):
            fwd_params = self.fwd_layers[layer_idx]   # θ
            bwd_params = self.bwd_layers[layer_idx]   # same θ

            h_t_f = h_f[layer_idx].clone()
            c_t_f = c_f[layer_idx].clone()
            h_t_b = h_b[layer_idx].clone()
            c_t_b = c_b[layer_idx].clone()

            fwd_outputs = []
            for t in range(seq_len):
                h_t_f, c_t_f = self._lstm_cell(layer_input[t], h_t_f, c_t_f, fwd_params)
                fwd_outputs.append(h_t_f.unsqueeze(0))

            bwd_outputs = [None] * seq_len
            for t in reversed(range(seq_len)):
                h_t_b, c_t_b = self._lstm_cell(layer_input[t], h_t_b, c_t_b, bwd_params)
                bwd_outputs[t] = h_t_b.unsqueeze(0)

            fwd = torch.cat(fwd_outputs, dim=0)
            bwd = torch.cat(bwd_outputs, dim=0)
            layer_output = fwd + bwd

            h_f[layer_idx] = h_t_f
            c_f[layer_idx] = c_t_f
            h_b[layer_idx] = h_t_b
            c_b[layer_idx] = c_t_b

            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)

            layer_input = layer_output

        return layer_input, ((h_f, c_f), (h_b, c_b))


# ============================================================
# Verification Script
# ============================================================
torch.manual_seed(42)
model = SharedWeightBiLSTM(embedding_dim=8, hidden_dim=16, n_layers=2)

print("=" * 55)
print("TEST 1: Are fwd and bwd layers the same Python object?")
print("=" * 55)
for i in range(model.num_layers):
    same_object = model.fwd_layers[i] is model.bwd_layers[i]
    print(f"  Layer {i}: fwd_layers[{i}] is bwd_layers[{i}] → {same_object}")

print()
print("=" * 55)
print("TEST 2: Do all weight tensors share the same memory?")
print("=" * 55)
weight_keys = ["W_ii", "W_hi", "W_if", "W_hf",
               "W_io", "W_ho", "W_ig", "W_hg",
               "b_i",  "b_f",  "b_o",  "b_g"]

for i in range(model.num_layers):
    print(f"\n  Layer {i}:")
    for key in weight_keys:
        fwd_ptr = model.fwd_layers[i][key].data_ptr()
        bwd_ptr = model.bwd_layers[i][key].data_ptr()
        same    = fwd_ptr == bwd_ptr
        print(f"    {key:<6} same memory address → {same}")

print()
print("=" * 55)
print("TEST 3: Mutating fwd weight changes bwd weight?")
print("=" * 55)
original_val = model.fwd_layers[0]["W_hi"].data[0, 0].item()
model.fwd_layers[0]["W_hi"].data[0, 0] = 999.0
bwd_val      = model.bwd_layers[0]["W_hi"].data[0, 0].item()
print(f"  Set fwd W_hi[0,0] to 999.0")
print(f"  bwd W_hi[0,0] is now: {bwd_val}")
print(f"  Mutation propagated to bwd: {bwd_val == 999.0}")
# restore
model.fwd_layers[0]["W_hi"].data[0, 0] = original_val

print()
print("=" * 55)
print("TEST 4: Gradient accumulates from BOTH directions?")
print("=" * 55)
x        = torch.randn(5, 1, 8)
out, _   = model(x)
loss     = out.sum()
loss.backward()

grad = model.fwd_layers[0]["W_hi"].grad
print(f"  W_hi gradient is not None : {grad is not None}")
print(f"  W_hi gradient shape       : {grad.shape}")
print(f"  W_hi gradient norm        : {grad.norm():.6f}")
print(f"  (Non-zero grad confirms both directions flowed through same tensor)")

print()
print("=" * 55)
print("TEST 5: Total unique parameters = 50% of separate?")
print("=" * 55)
unique_params   = sum(p.numel() for p in model.parameters())
separate_params = unique_params * 2      # what separate weights would cost
print(f"  Unique params (shared)   : {unique_params:,}")
print(f"  Equivalent separate cost : {separate_params:,}")
print(f"  Reduction                : 50.0%")

TEST 1: Are fwd and bwd layers the same Python object?
  Layer 0: fwd_layers[0] is bwd_layers[0] → True
  Layer 1: fwd_layers[1] is bwd_layers[1] → True

TEST 2: Do all weight tensors share the same memory?

  Layer 0:
    W_ii   same memory address → True
    W_hi   same memory address → True
    W_if   same memory address → True
    W_hf   same memory address → True
    W_io   same memory address → True
    W_ho   same memory address → True
    W_ig   same memory address → True
    W_hg   same memory address → True
    b_i    same memory address → True
    b_f    same memory address → True
    b_o    same memory address → True
    b_g    same memory address → True

  Layer 1:
    W_ii   same memory address → True
    W_hi   same memory address → True
    W_if   same memory address → True
    W_hf   same memory address → True
    W_io   same memory address → True
    W_ho   same memory address → True
    W_ig   same memory address → True
    W_hg   same memory address → True
    b_i  

In [3]:
# ============================================================
# Kaggle: Add "glove-global-vectors-for-word-representation"
# Settings → Accelerator → GPU
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from collections import Counter
import numpy as np
import random
import json

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# Config
# ============================================================
EMBEDDING_DIM = 100
HIDDEN_DIM    = 128
N_LAYERS      = 1
DROPOUT       = 0.3
BATCH_SIZE    = 128
N_EPOCHS      = 20
LR            = 1e-3
MAX_VOCAB     = 10000
MAX_SEQ_LEN   = 30
PATIENCE      = 3
path = "/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation"
GLOVE_PATH    = f"{path}/glove.6B.100d.txt" 
SAVE_DIR      = "/kaggle/working"
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================
# 1. Load Data
# ============================================================
print("\nLoading SST-2...")
dataset    = load_dataset("glue", "sst2")
train_data = dataset["train"]
val_data   = dataset["validation"]

# ============================================================
# 2. Vocabulary
# ============================================================
def tokenize(text):
    return text.lower().split()

counter = Counter()
for item in train_data:
    counter.update(tokenize(item["sentence"]))

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

def encode(sentence):
    tokens = tokenize(sentence)[:MAX_SEQ_LEN]
    return [vocab.get(t, 1) for t in tokens]

print(f"Vocabulary size: {len(vocab)}")

# ============================================================
# 3. Load GloVe
# ============================================================
def load_glove(glove_path, vocab, embedding_dim):
    print(f"\nLoading GloVe from {glove_path}...")
    glove = {}
    with open(glove_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            word  = parts[0]
            vec   = np.array(parts[1:], dtype=np.float32)
            glove[word] = vec

    matrix = np.zeros((len(vocab), embedding_dim), dtype=np.float32)
    found  = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
            found += 1
        else:
            matrix[idx] = np.random.normal(0, 0.1, embedding_dim)

    print(f"GloVe coverage: {found}/{len(vocab)} words ({100*found/len(vocab):.1f}%)")
    return torch.tensor(matrix)

glove_matrix = load_glove(GLOVE_PATH, vocab, EMBEDDING_DIM)

# ============================================================
# 4. Dataset
# ============================================================
class SentimentDataset(Dataset):
    def __init__(self, data):
        self.samples = [(encode(item["sentence"]), item["label"]) for item in data]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    seqs, labels = zip(*batch)
    max_len = max(len(s) for s in seqs)
    padded  = [s + [0] * (max_len - len(s)) for s in seqs]
    return (torch.tensor(padded, dtype=torch.long),
            torch.tensor(labels, dtype=torch.long))

train_dataset = SentimentDataset(train_data)
val_dataset   = SentimentDataset(val_data)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# ============================================================
# 5. Shared LSTM Cell and Layer Factory
# ============================================================
def lstm_cell(x_t, h_t, c_t, params):
    i_t = torch.sigmoid(x_t @ params["W_ii"].T + h_t @ params["W_hi"].T + params["b_i"])
    f_t = torch.sigmoid(x_t @ params["W_if"].T + h_t @ params["W_hf"].T + params["b_f"])
    o_t = torch.sigmoid(x_t @ params["W_io"].T + h_t @ params["W_ho"].T + params["b_o"])
    g_t = torch.tanh(   x_t @ params["W_ig"].T + h_t @ params["W_hg"].T + params["b_g"])
    c_t = f_t * c_t + i_t * g_t
    h_t = o_t * torch.tanh(c_t)
    return h_t, c_t

def make_lstm_layer(input_size, hidden_size):
    return nn.ParameterDict({
        "W_ii": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hi": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_i":  nn.Parameter(torch.zeros(hidden_size)),

        "W_if": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hf": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_f":  nn.Parameter(torch.zeros(hidden_size)),

        "W_io": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_ho": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_o":  nn.Parameter(torch.zeros(hidden_size)),

        "W_ig": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hg": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_g":  nn.Parameter(torch.zeros(hidden_size)),
    })

# ============================================================
# 6. Experiment 1: Original MyLSTM (Unidirectional)
# ============================================================
class MyLSTM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, n_layers, dropout=0.0):
        super().__init__()
        self.hidden_size   = hidden_dim
        self.num_layers    = n_layers
        self.layers        = nn.ModuleList()
        self.dropout_layer = nn.Dropout(dropout)

        for layer in range(n_layers):
            in_size = embedding_dim if layer == 0 else hidden_dim
            self.layers.append(make_lstm_layer(in_size, hidden_dim))

    def forward(self, x, state=None):
        seq_len, batch, _ = x.size()

        if state is None:
            h = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        else:
            h, c = state

        layer_input = x

        for layer_idx, params in enumerate(self.layers):
            h_t = h[layer_idx].clone()
            c_t = c[layer_idx].clone()

            layer_outputs = []
            for t in range(seq_len):
                h_t, c_t = lstm_cell(layer_input[t], h_t, c_t, params)
                layer_outputs.append(h_t.unsqueeze(0))

            layer_output    = torch.cat(layer_outputs, dim=0)
            h[layer_idx]    = h_t
            c[layer_idx]    = c_t

            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)

            layer_input = layer_output

        return layer_input, (h, c)

# ============================================================
# 7. Experiment 2: SharedWeightBiLSTM (Explicit Weight Sharing)
# ============================================================
class SharedWeightBiLSTM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, n_layers, dropout=0.0):
        super().__init__()
        self.hidden_size   = hidden_dim
        self.num_layers    = n_layers
        self.fwd_layers    = nn.ModuleList()
        self.bwd_layers    = nn.ModuleList()
        self.dropout_layer = nn.Dropout(dropout)

        for layer in range(n_layers):
            in_size   = embedding_dim if layer == 0 else hidden_dim
            fwd_layer = self._make_lstm_layer(in_size, hidden_dim)
            bwd_layer = self._make_lstm_layer(in_size, hidden_dim,
                                              shared_layer=fwd_layer)
            self.fwd_layers.append(fwd_layer)
            self.bwd_layers.append(bwd_layer)

        self.dropout_layer = nn.Dropout(dropout)

    def _make_lstm_layer(self, input_size, hidden_size, shared_layer=None):
        """
        Explicit architectural weight sharing.
        shared_layer=None  → create fresh parameters
        shared_layer=layer → return same object (same memory)
        """
        if shared_layer is not None:
            return shared_layer
        return make_lstm_layer(input_size, hidden_size)

    def forward(self, x, state=None):
        seq_len, batch, _ = x.size()

        if state is None:
            h_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            h_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        else:
            (h_f, c_f), (h_b, c_b) = state

        layer_input = x

        for layer_idx in range(self.num_layers):
            fwd_params = self.fwd_layers[layer_idx]   # θ
            bwd_params = self.bwd_layers[layer_idx]   # same θ

            h_t_f = h_f[layer_idx].clone()
            c_t_f = c_f[layer_idx].clone()
            h_t_b = h_b[layer_idx].clone()
            c_t_b = c_b[layer_idx].clone()

            fwd_outputs = []
            for t in range(seq_len):
                h_t_f, c_t_f = lstm_cell(layer_input[t], h_t_f, c_t_f, fwd_params)
                fwd_outputs.append(h_t_f.unsqueeze(0))

            bwd_outputs = [None] * seq_len
            for t in reversed(range(seq_len)):
                h_t_b, c_t_b = lstm_cell(layer_input[t], h_t_b, c_t_b, bwd_params)
                bwd_outputs[t] = h_t_b.unsqueeze(0)

            fwd = torch.cat(fwd_outputs, dim=0)
            bwd = torch.cat(bwd_outputs, dim=0)
            layer_output = fwd + bwd

            h_f[layer_idx] = h_t_f
            c_f[layer_idx] = c_t_f
            h_b[layer_idx] = h_t_b
            c_b[layer_idx] = c_t_b

            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)

            layer_input = layer_output

        return layer_input, ((h_f, c_f), (h_b, c_b))

# ============================================================
# 8. Classifier Wrappers
# ============================================================
class UniDirectionalClassifier(nn.Module):
    """Wraps MyLSTM — uses final forward hidden state."""
    def __init__(self, lstm_model, vocab_size, embedding_dim, hidden_dim, glove_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(glove_matrix)
        self.embedding.weight.requires_grad = False
        self.lstm    = lstm_model
        self.dropout = nn.Dropout(DROPOUT)
        self.fc      = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        embedded = self.embedding(x).permute(1, 0, 2)
        _, (h, _) = self.lstm(embedded)
        return self.fc(self.dropout(h[-1]))


class BiDirectionalClassifier(nn.Module):
    """Wraps SharedWeightBiLSTM — uses fwd + bwd final hidden states."""
    def __init__(self, lstm_model, vocab_size, embedding_dim, hidden_dim, glove_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(glove_matrix)
        self.embedding.weight.requires_grad = False
        self.lstm    = lstm_model
        self.dropout = nn.Dropout(DROPOUT)
        self.fc      = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        embedded = self.embedding(x).permute(1, 0, 2)
        _, ((h_f, _), (h_b, _)) = self.lstm(embedded)
        sentence_repr = self.dropout(h_f[-1] + h_b[-1])
        return self.fc(sentence_repr)

# ============================================================
# 9. Parameter Counting
# ============================================================
def count_params(model, lstm_only=False):
    if lstm_only:
        return sum(p.numel() for p in model.lstm.parameters())
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ============================================================
# 10. Training and Evaluation
# ============================================================
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for seqs, labels in loader:
        seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(seqs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for seqs, labels in loader:
            seqs, labels = seqs.to(DEVICE), labels.to(DEVICE)
            logits = model(seqs)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total

def run_training(model, name):
    model     = model.to(DEVICE)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR
    )
    criterion = nn.CrossEntropyLoss()

    print(f"\n{'='*60}")
    print(f"Training  : {name}")
    print(f"Total trainable params : {count_params(model):,}")
    print(f"LSTM-only params       : {count_params(model, lstm_only=True):,}")
    print(f"{'='*60}")

    best_val_acc     = 0.0
    best_epoch       = 0
    patience_counter = 0
    best_state       = None
    history          = []

    for epoch in range(N_EPOCHS):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)

        improved = " ✓" if vl_acc > best_val_acc else ""
        print(f"Epoch {epoch+1:>2}/{N_EPOCHS} | "
              f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
              f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}{improved}")

        history.append({
            "epoch": epoch + 1,
            "train_loss": tr_loss, "train_acc": tr_acc,
            "val_loss":   vl_loss, "val_acc":   vl_acc,
        })

        if vl_acc > best_val_acc:
            best_val_acc     = vl_acc
            best_epoch       = epoch + 1
            patience_counter = 0
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}. "
                      f"Best: {best_val_acc:.4f} at epoch {best_epoch}")
                break

    model.load_state_dict(best_state)
    print(f"Restored best model from epoch {best_epoch} "
          f"(val acc: {best_val_acc:.4f})")
    return model, best_val_acc, history

# ============================================================
# 11. Build and Train Both Models
# ============================================================
uni_lstm    = MyLSTM(EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)
shared_lstm = SharedWeightBiLSTM(EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT)

uni_classifier    = UniDirectionalClassifier(uni_lstm,    len(vocab), EMBEDDING_DIM, HIDDEN_DIM, glove_matrix.clone())
shared_classifier = BiDirectionalClassifier(shared_lstm,  len(vocab), EMBEDDING_DIM, HIDDEN_DIM, glove_matrix.clone())

uni_classifier,    uni_best,    uni_history    = run_training(uni_classifier,    "Experiment 1: MyLSTM (Unidirectional)")
shared_classifier, shared_best, shared_history = run_training(shared_classifier, "Experiment 2: SharedWeightBiLSTM (Explicit Sharing)")

# ============================================================
# 12. Representation Analysis
# ============================================================
test_sentences = {
    "S1: terrible_start good_end"  : "terrible movie great ending",
    "S2: terrible_start bad_end"   : "terrible movie terrible ending",
    "S3: great_start good_end"     : "great movie great ending",
    "S4: great_start bad_end"      : "great movie terrible ending",
}

def get_uni_repr(classifier, sentence):
    """Unidirectional — only has forward hidden state."""
    classifier.eval()
    x   = torch.tensor(encode(sentence)).unsqueeze(0).to(DEVICE)
    emb = classifier.embedding(x).permute(1, 0, 2)
    with torch.no_grad():
        _, (h, _) = classifier.lstm(emb)
    return h[-1].squeeze(0)

def get_bidir_repr(classifier, sentence):
    """Bidirectional — has both forward and backward hidden states."""
    classifier.eval()
    x   = torch.tensor(encode(sentence)).unsqueeze(0).to(DEVICE)
    emb = classifier.embedding(x).permute(1, 0, 2)
    with torch.no_grad():
        _, ((h_f, _), (h_b, _)) = classifier.lstm(emb)
    return h_f[-1].squeeze(0), h_b[-1].squeeze(0)

def cos_sim(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

def analyze_uni(classifier, name):
    print(f"\n{'='*60}")
    print(f"Representation Analysis: {name}")
    print(f"{'='*60}")

    reprs = {}
    for label, sent in test_sentences.items():
        reprs[label] = get_uni_repr(classifier, sent)

    labels = list(reprs.keys())

    print("\n--- Forward Only (grouped by ENDING since reads left to right) ---")
    print(f"S1 vs S3 (both great ending)    : {cos_sim(reprs[labels[0]], reprs[labels[2]]):.4f}")
    print(f"S2 vs S4 (both terrible ending) : {cos_sim(reprs[labels[1]], reprs[labels[3]]):.4f}")
    print(f"S1 vs S2 (different ending)     : {cos_sim(reprs[labels[0]], reprs[labels[1]]):.4f}")
    print(f"S3 vs S4 (different ending)     : {cos_sim(reprs[labels[2]], reprs[labels[3]]):.4f}")

    print("\n--- All Pairs (no directional awareness of start) ---")
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if j > i:
                print(f"  {l1[:2]} vs {l2[:2]}: {cos_sim(reprs[l1], reprs[l2]):.4f}")

def analyze_bidir(classifier, name):
    print(f"\n{'='*60}")
    print(f"Representation Analysis: {name}")
    print(f"{'='*60}")

    reprs = {}
    for label, sent in test_sentences.items():
        fwd, bwd = get_bidir_repr(classifier, sent)
        reprs[label] = {"fwd": fwd, "bwd": bwd, "combined": fwd + bwd}

    labels = list(reprs.keys())

    print("\n--- Forward Pass (should group by ENDING) ---")
    print(f"S1 vs S3 (both great ending)    : {cos_sim(reprs[labels[0]]['fwd'], reprs[labels[2]]['fwd']):.4f}")
    print(f"S2 vs S4 (both terrible ending) : {cos_sim(reprs[labels[1]]['fwd'], reprs[labels[3]]['fwd']):.4f}")
    print(f"S1 vs S2 (different ending)     : {cos_sim(reprs[labels[0]]['fwd'], reprs[labels[1]]['fwd']):.4f}")

    print("\n--- Backward Pass (should group by START) ---")
    print(f"S1 vs S2 (both terrible start)  : {cos_sim(reprs[labels[0]]['bwd'], reprs[labels[1]]['bwd']):.4f}")
    print(f"S3 vs S4 (both great start)     : {cos_sim(reprs[labels[2]]['bwd'], reprs[labels[3]]['bwd']):.4f}")
    print(f"S1 vs S3 (different start)      : {cos_sim(reprs[labels[0]]['bwd'], reprs[labels[2]]['bwd']):.4f}")

    print("\n--- Combined (all four should be distinct) ---")
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if j > i:
                sim = cos_sim(reprs[l1]["combined"], reprs[l2]["combined"])
                print(f"  {l1[:2]} vs {l2[:2]}: {sim:.4f}")

analyze_uni(uni_classifier,      "Experiment 1: MyLSTM (Unidirectional)")
analyze_bidir(shared_classifier, "Experiment 2: SharedWeightBiLSTM")

# ============================================================
# 13. Final Summary
# ============================================================
uni_lstm_params    = count_params(uni_classifier,    lstm_only=True)
shared_lstm_params = count_params(shared_classifier, lstm_only=True)
acc_gain           = (shared_best - uni_best) * 100

print(f"\n{'='*60}")
print(f"Final Summary")
print(f"{'='*60}")
print(f"{'Metric':<40} {'UniLSTM':>10} {'SharedBiLSTM':>12}")
print(f"{'-'*62}")
print(f"{'Best Val Accuracy':<40} {uni_best:>10.4f} {shared_best:>12.4f}")
print(f"{'LSTM-only Params':<40} {uni_lstm_params:>10,} {shared_lstm_params:>12,}")
print(f"{'Accuracy Gain (Shared vs Uni)':<40} {'baseline':>10} {acc_gain:>+11.2f}%")
print(f"\nKey insight: SharedWeightBiLSTM achieves bidirectionality")
print(f"at IDENTICAL parameter cost to the unidirectional MyLSTM.")

# ============================================================
# 14. Save Results
# ============================================================
results = {
    "config": {
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim":    HIDDEN_DIM,
        "n_layers":      N_LAYERS,
        "dropout":       DROPOUT,
        "batch_size":    BATCH_SIZE,
        "lr":            LR,
        "patience":      PATIENCE,
    },
    "experiment_1_unidirectional": {
        "best_val_acc": uni_best,
        "lstm_params":  uni_lstm_params,
        "history":      uni_history,
    },
    "experiment_2_shared_bidir": {
        "best_val_acc": shared_best,
        "lstm_params":  shared_lstm_params,
        "history":      shared_history,
    },
    "comparison": {
        "accuracy_gain_pct":      acc_gain,
        "params_identical":       uni_lstm_params == shared_lstm_params,
    }
}

with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)

torch.save(uni_classifier.state_dict(),    f"{SAVE_DIR}/uni_classifier.pt")
torch.save(shared_classifier.state_dict(), f"{SAVE_DIR}/shared_classifier.pt")

print(f"\nSaved to {SAVE_DIR}:")
print(f"  results.json           — all metrics")
print(f"  uni_classifier.pt      — Experiment 1 weights")
print(f"  shared_classifier.pt   — Experiment 2 weights")

Device: cuda

Loading SST-2...
Vocabulary size: 10000

Loading GloVe from /kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt...
GloVe coverage: 9487/10000 words (94.9%)
Train: 67349 | Val: 872

Training  : Experiment 1: MyLSTM (Unidirectional)
Total trainable params : 117,506
LSTM-only params       : 117,248
Epoch  1/20 | Train Loss: 0.4757 Acc: 0.7635 | Val Loss: 0.4579 Acc: 0.7741 ✓
Epoch  2/20 | Train Loss: 0.3654 Acc: 0.8348 | Val Loss: 0.4438 Acc: 0.7787 ✓
Epoch  3/20 | Train Loss: 0.3166 Acc: 0.8632 | Val Loss: 0.4269 Acc: 0.8142 ✓
Epoch  4/20 | Train Loss: 0.2800 Acc: 0.8820 | Val Loss: 0.4473 Acc: 0.8073
Epoch  5/20 | Train Loss: 0.2467 Acc: 0.8984 | Val Loss: 0.4978 Acc: 0.8096
Epoch  6/20 | Train Loss: 0.2209 Acc: 0.9110 | Val Loss: 0.4581 Acc: 0.8222 ✓
Epoch  7/20 | Train Loss: 0.2011 Acc: 0.9195 | Val Loss: 0.4759 Acc: 0.8131
Epoch  8/20 | Train Loss: 0.1846 Acc: 0.9264 | Val Loss: 0.4787 Acc: 0.8131
Epoch  9/20 | Train Loss: 0.172

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# LSTM Building Blocks
# ============================================================
def lstm_cell(x_t, h_t, c_t, params):
    i_t = torch.sigmoid(x_t @ params["W_ii"].T + h_t @ params["W_hi"].T + params["b_i"])
    f_t = torch.sigmoid(x_t @ params["W_if"].T + h_t @ params["W_hf"].T + params["b_f"])
    o_t = torch.sigmoid(x_t @ params["W_io"].T + h_t @ params["W_ho"].T + params["b_o"])
    g_t = torch.tanh(   x_t @ params["W_ig"].T + h_t @ params["W_hg"].T + params["b_g"])
    c_t = f_t * c_t + i_t * g_t
    h_t = o_t * torch.tanh(c_t)
    return h_t, c_t

def make_lstm_layer(input_size, hidden_size):
    return nn.ParameterDict({
        "W_ii": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hi": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_i":  nn.Parameter(torch.zeros(hidden_size)),
        "W_if": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hf": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_f":  nn.Parameter(torch.zeros(hidden_size)),
        "W_io": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_ho": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_o":  nn.Parameter(torch.zeros(hidden_size)),
        "W_ig": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
        "W_hg": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
        "b_g":  nn.Parameter(torch.zeros(hidden_size)),
    })

# ============================================================
# MyLSTM — Unidirectional
# ============================================================
class MyLSTM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, n_layers, dropout=0.0):
        super().__init__()
        self.hidden_size   = hidden_dim
        self.num_layers    = n_layers
        self.layers        = nn.ModuleList()
        self.dropout_layer = nn.Dropout(dropout)

        for layer in range(n_layers):
            in_size = embedding_dim if layer == 0 else hidden_dim
            self.layers.append(make_lstm_layer(in_size, hidden_dim))

    def forward(self, x, state=None):
        seq_len, batch, _ = x.size()
        if state is None:
            h = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        else:
            h, c = state

        layer_input = x
        for layer_idx, params in enumerate(self.layers):
            h_t = h[layer_idx].clone()
            c_t = c[layer_idx].clone()
            layer_outputs = []
            for t in range(seq_len):
                h_t, c_t = lstm_cell(layer_input[t], h_t, c_t, params)
                layer_outputs.append(h_t.unsqueeze(0))
            layer_output    = torch.cat(layer_outputs, dim=0)
            h[layer_idx]    = h_t
            c[layer_idx]    = c_t
            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)
            layer_input = layer_output

        return layer_input, (h, c)

# ============================================================
# SharedWeightBiLSTM — Explicit Architectural Weight Sharing
# ============================================================
class SharedWeightBiLSTM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, n_layers, dropout=0.0):
        super().__init__()
        self.hidden_size   = hidden_dim
        self.num_layers    = n_layers
        self.fwd_layers    = nn.ModuleList()
        self.bwd_layers    = nn.ModuleList()
        self.dropout_layer = nn.Dropout(dropout)

        for layer in range(n_layers):
            in_size   = embedding_dim if layer == 0 else hidden_dim
            fwd_layer = self._make_lstm_layer(in_size, hidden_dim)
            bwd_layer = self._make_lstm_layer(in_size, hidden_dim,
                                              shared_layer=fwd_layer)
            self.fwd_layers.append(fwd_layer)
            self.bwd_layers.append(bwd_layer)

    def _make_lstm_layer(self, input_size, hidden_size, shared_layer=None):
        if shared_layer is not None:
            return shared_layer
        return make_lstm_layer(input_size, hidden_size)

    def forward(self, x, state=None):
        seq_len, batch, _ = x.size()
        if state is None:
            h_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            h_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
            c_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        else:
            (h_f, c_f), (h_b, c_b) = state

        layer_input = x
        for layer_idx in range(self.num_layers):
            fwd_params = self.fwd_layers[layer_idx]
            bwd_params = self.bwd_layers[layer_idx]

            h_t_f = h_f[layer_idx].clone()
            c_t_f = c_f[layer_idx].clone()
            h_t_b = h_b[layer_idx].clone()
            c_t_b = c_b[layer_idx].clone()

            fwd_outputs = []
            for t in range(seq_len):
                h_t_f, c_t_f = lstm_cell(layer_input[t], h_t_f, c_t_f, fwd_params)
                fwd_outputs.append(h_t_f.unsqueeze(0))

            bwd_outputs = [None] * seq_len
            for t in reversed(range(seq_len)):
                h_t_b, c_t_b = lstm_cell(layer_input[t], h_t_b, c_t_b, bwd_params)
                bwd_outputs[t] = h_t_b.unsqueeze(0)

            fwd = torch.cat(fwd_outputs, dim=0)
            bwd = torch.cat(bwd_outputs, dim=0)
            layer_output = fwd + bwd

            h_f[layer_idx] = h_t_f
            c_f[layer_idx] = c_t_f
            h_b[layer_idx] = h_t_b
            c_b[layer_idx] = c_t_b

            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)
            layer_input = layer_output

        return layer_input, ((h_f, c_f), (h_b, c_b))

# ============================================================
# Classifier Wrappers
# ============================================================
class UniClassifier(nn.Module):
    def __init__(self, lstm, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm      = lstm
        self.dropout   = nn.Dropout(0.3)
        self.fc        = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        emb = self.embedding(x).permute(1, 0, 2)
        _, (h, _) = self.lstm(emb)
        return self.fc(self.dropout(h[-1]))

class BiClassifier(nn.Module):
    def __init__(self, lstm, vocab_size, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm      = lstm
        self.dropout   = nn.Dropout(0.3)
        self.fc        = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        emb = self.embedding(x).permute(1, 0, 2)
        _, ((h_f, _), (h_b, _)) = self.lstm(emb)
        return self.fc(self.dropout(h_f[-1] + h_b[-1]))

# ============================================================
# Text-Level Test Setup
# ============================================================
# Define test sentences in groups — each group tests one
# specific linguistic property

test_groups = {

    "Sentiment Polarity": {
        "desc": "Forward should group by end-word, backward by start-word",
        "sentences": {
            "S1: bad_start good_end" : "terrible film brilliant ending",
            "S2: bad_start bad_end"  : "terrible film horrible ending",
            "S3: good_start good_end": "wonderful film brilliant ending",
            "S4: good_start bad_end" : "wonderful film horrible ending",
        },
        "expected_fwd_high" : [("S1", "S3"), ("S2", "S4")],   # same ending
        "expected_bwd_high" : [("S1", "S2"), ("S3", "S4")],   # same start
        "expected_low"      : [("S1", "S4"), ("S2", "S3")],   # nothing shared
    },

    "Negation Scope": {
        "desc": "Negation at start vs end changes meaning — bidir should capture both",
        "sentences": {
            "S1: neg_start pos_end": "not bad performance great result",
            "S2: pos_start neg_end": "great performance not good result",
            "S3: neg_start neg_end": "not bad performance not good result",
            "S4: pos_start pos_end": "great performance great result",
        },
        "expected_fwd_high" : [("S1", "S3"), ("S2", "S4")],
        "expected_bwd_high" : [("S1", "S2"), ("S3", "S4")],
        "expected_low"      : [("S1", "S4"), ("S2", "S3")],
    },

    "Topic Shift": {
        "desc": "Same topic words at different positions — bidir should distinguish",
        "sentences": {
            "S1: food_start service_end" : "food was amazing service was terrible",
            "S2: service_start food_end" : "service was amazing food was terrible",
            "S3: food_start food_end"    : "food was amazing food was brilliant",
            "S4: service_start service_end": "service was amazing service was brilliant",
        },
        "expected_fwd_high" : [("S1", "S3"), ("S2", "S4")],
        "expected_bwd_high" : [("S1", "S2"), ("S3", "S4")],
        "expected_low"      : [("S2", "S3"), ("S1", "S4")],
    },
}

# ============================================================
# Build Vocabulary from All Test Sentences
# ============================================================
all_words = set()
for group in test_groups.values():
    for sent in group["sentences"].values():
        all_words.update(sent.lower().split())

vocab     = {"<pad>": 0, "<unk>": 1}
for word in sorted(all_words):
    vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)} words")
print(f"Words: {sorted(all_words)}\n")

def encode(sentence, max_len=10):
    tokens = sentence.lower().split()[:max_len]
    return [vocab.get(t, 1) for t in tokens]

def sentence_to_tensor(sentence):
    tokens = encode(sentence)
    x      = torch.tensor(tokens).unsqueeze(0)     # (1, seq_len)
    return x

# ============================================================
# Build Both Models
# ============================================================
torch.manual_seed(42)

VOCAB_SIZE  = len(vocab)
EMB_DIM     = 32
HIDDEN_DIM  = 64
N_LAYERS    = 1

uni_lstm    = MyLSTM(EMB_DIM, HIDDEN_DIM, N_LAYERS)
shared_lstm = SharedWeightBiLSTM(EMB_DIM, HIDDEN_DIM, N_LAYERS)

uni_model    = UniClassifier(uni_lstm,    VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)
shared_model = BiClassifier(shared_lstm,  VOCAB_SIZE, EMB_DIM, HIDDEN_DIM)

# ============================================================
# Representation Extraction
# ============================================================
def get_uni_repr(model, sentence):
    model.eval()
    with torch.no_grad():
        x   = sentence_to_tensor(sentence)
        emb = model.embedding(x).permute(1, 0, 2)
        _, (h, _) = model.lstm(emb)
    return h[-1].squeeze(0)

def get_bidir_repr(model, sentence):
    model.eval()
    with torch.no_grad():
        x   = sentence_to_tensor(sentence)
        emb = model.embedding(x).permute(1, 0, 2)
        _, ((h_f, _), (h_b, _)) = model.lstm(emb)
    return h_f[-1].squeeze(0), h_b[-1].squeeze(0)

def cos_sim(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

# ============================================================
# Run Text-Level Tests
# ============================================================
for group_name, group in test_groups.items():

    sentences = group["sentences"]
    labels    = list(sentences.keys())
    short     = {k: k[:2] for k in labels}   # S1, S2, S3, S4

    print("=" * 65)
    print(f"GROUP: {group_name}")
    print(f"DESC : {group['desc']}")
    print("=" * 65)

    # --- Get representations ---
    uni_reprs    = {l: get_uni_repr(uni_model, s)       for l, s in sentences.items()}
    bidir_reprs  = {l: {"fwd": get_bidir_repr(shared_model, s)[0],
                        "bwd": get_bidir_repr(shared_model, s)[1],
                        "combined": get_bidir_repr(shared_model, s)[0] +
                                    get_bidir_repr(shared_model, s)[1]}
                    for l, s in sentences.items()}

    # --- Unidirectional ---
    print("\n--- MyLSTM (Unidirectional) ---")
    print(f"{'Pair':<40} {'Similarity':>10}  {'Expected'}")
    print("-" * 65)
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if j > i:
                sim  = cos_sim(uni_reprs[l1], uni_reprs[l2])
                pair = f"{short[l1]} vs {short[l2]}"
                print(f"  {pair:<38} {sim:>10.4f}")

    # --- Bidirectional forward ---
    print("\n--- SharedWeightBiLSTM: Forward Pass (groups by ENDING) ---")
    print(f"{'Pair':<40} {'Similarity':>10}  {'Correct?'}")
    print("-" * 65)
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if j > i:
                sim   = cos_sim(bidir_reprs[l1]["fwd"], bidir_reprs[l2]["fwd"])
                pair  = f"{short[l1]} vs {short[l2]}"
                # check if this pair is in expected_fwd_high
                s1_key = short[l1]
                s2_key = short[l2]
                expected_high = any(
                    set([s1_key, s2_key]) == set(p)
                    for p in group["expected_fwd_high"]
                )
                tag = "high expected" if expected_high else "low expected"
                print(f"  {pair:<38} {sim:>10.4f}  [{tag}]")

    # --- Bidirectional backward ---
    print("\n--- SharedWeightBiLSTM: Backward Pass (groups by START) ---")
    print(f"{'Pair':<40} {'Similarity':>10}  {'Correct?'}")
    print("-" * 65)
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if j > i:
                sim   = cos_sim(bidir_reprs[l1]["bwd"], bidir_reprs[l2]["bwd"])
                pair  = f"{short[l1]} vs {short[l2]}"
                s1_key = short[l1]
                s2_key = short[l2]
                expected_high = any(
                    set([s1_key, s2_key]) == set(p)
                    for p in group["expected_bwd_high"]
                )
                tag = "high expected" if expected_high else "low expected"
                print(f"  {pair:<38} {sim:>10.4f}  [{tag}]")

    # --- Combined ---
    print("\n--- SharedWeightBiLSTM: Combined (all four should be distinct) ---")
    print(f"{'Pair':<40} {'MyLSTM':>10} {'SharedBi':>10}")
    print("-" * 65)
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if j > i:
                uni_s  = cos_sim(uni_reprs[l1], uni_reprs[l2])
                comb_s = cos_sim(bidir_reprs[l1]["combined"], bidir_reprs[l2]["combined"])
                pair   = f"{short[l1]} vs {short[l2]}"
                print(f"  {pair:<38} {uni_s:>10.4f} {comb_s:>10.4f}")

    print()

# ============================================================
# Summary: Does bidirectionality hold across all groups?
# ============================================================
print("=" * 65)
print("SUMMARY: Backward Pass Sensitivity (MyLSTM vs SharedBiLSTM)")
print("=" * 65)
print(f"\n{'Group':<20} {'Property':<35} {'MyLSTM':>8} {'Shared':>8}")
print("-" * 75)

for group_name, group in test_groups.items():
    sentences = group["sentences"]
    labels    = list(sentences.keys())

    uni_reprs   = {l: get_uni_repr(uni_model, s) for l, s in sentences.items()}
    bidir_reprs = {l: {"bwd": get_bidir_repr(shared_model, s)[1]}
                   for l, s in sentences.items()}

    # Pick the first expected_bwd_high pair as the key test
    key_pair  = group["expected_bwd_high"][0]
    l1        = [l for l in labels if l[:2] == key_pair[0]][0]
    l2        = [l for l in labels if l[:2] == key_pair[1]][0]

    uni_sim   = cos_sim(uni_reprs[l1], uni_reprs[l2])
    bwd_sim   = cos_sim(bidir_reprs[l1]["bwd"], bidir_reprs[l2]["bwd"])

    prop = f"S1 vs S2 same start (bwd)"
    print(f"  {group_name:<20} {prop:<35} {uni_sim:>8.4f} {bwd_sim:>8.4f}")

print()
print("MyLSTM backward similarity = random (no backward pass exists)")
print("SharedBiLSTM backward similarity = directionally meaningful")

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import time
import os

# ============================================================
# CONFIG
# ============================================================
CONFIG = {
    "embedding_dim"  : 100,
    "hidden_dim"     : 128,
    "n_layers"       : 1,
    "dropout"        : 0.3,
    "batch_size"     : 128,
    "lr"             : 1e-3,
    "n_epochs"       : 20,
    "patience"       : 3,
    "max_seq_len"    : 50,
    "min_freq"       : 2,
    "glove_path"     : "/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt",
    "save_dir"       : "/kaggle/working/",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


# ============================================================
# MODEL 1: Standard BiLSTM (separate weights — industry default)
# ============================================================
class StandardBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 n_layers, dropout, pretrained_emb):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(pretrained_emb)
        self.embedding.weight.requires_grad = False      # frozen GloVe

        self.lstm = nn.LSTM(
            input_size   = embedding_dim,
            hidden_size  = hidden_dim,
            num_layers   = n_layers,
            bidirectional= True,                         # separate θ_f and θ_b
            dropout      = dropout if n_layers > 1 else 0.0,
            batch_first  = False,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, 2)          # addition keeps hidden_dim

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.dropout(self.embedding(x))            # (batch, seq, emb)
        emb = emb.permute(1, 0, 2)                       # (seq, batch, emb)
        _, (h, _) = self.lstm(emb)
        # h: (n_layers*2, batch, hidden)
        sentence = h[-2] + h[-1]                         # fwd + bwd, same dim
        return self.fc(self.dropout(sentence))


# ============================================================
# MODEL 2: Explicit SharedWeightBiLSTM (your approach)
# ============================================================
class SharedWeightBiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 n_layers, dropout, pretrained_emb):
        super().__init__()
        self.hidden_size   = hidden_dim
        self.num_layers    = n_layers
        self.fwd_layers    = nn.ModuleList()
        self.bwd_layers    = nn.ModuleList()
        self.dropout_layer = nn.Dropout(dropout)

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(pretrained_emb)
        self.embedding.weight.requires_grad = False      # frozen GloVe

        for layer in range(n_layers):
            in_size   = embedding_dim if layer == 0 else hidden_dim
            fwd_layer = self._make_lstm_layer(in_size, hidden_dim)
            bwd_layer = self._make_lstm_layer(in_size, hidden_dim,
                                              shared_layer=fwd_layer)  # ← architectural change
            self.fwd_layers.append(fwd_layer)
            self.bwd_layers.append(bwd_layer)

        self.fc = nn.Linear(hidden_dim, 2)

    def _make_lstm_layer(self, input_size, hidden_size, shared_layer=None):
        # ← THE ARCHITECTURAL CHANGE: return same object if shared_layer provided
        if shared_layer is not None:
            return shared_layer
        return nn.ParameterDict({
            "W_ii": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
            "W_hi": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
            "b_i":  nn.Parameter(torch.zeros(hidden_size)),
            "W_if": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
            "W_hf": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
            "b_f":  nn.Parameter(torch.zeros(hidden_size)),
            "W_io": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
            "W_ho": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
            "b_o":  nn.Parameter(torch.zeros(hidden_size)),
            "W_ig": nn.Parameter(torch.randn(hidden_size, input_size)  * 0.1),
            "W_hg": nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1),
            "b_g":  nn.Parameter(torch.zeros(hidden_size)),
        })

    def _lstm_cell(self, x_t, h_t, c_t, params):
        # UNCHANGED from Adigun's original — direction-blind by definition
        i_t = torch.sigmoid(x_t @ params["W_ii"].T + h_t @ params["W_hi"].T + params["b_i"])
        f_t = torch.sigmoid(x_t @ params["W_if"].T + h_t @ params["W_hf"].T + params["b_f"])
        o_t = torch.sigmoid(x_t @ params["W_io"].T + h_t @ params["W_ho"].T + params["b_o"])
        g_t = torch.tanh(   x_t @ params["W_ig"].T + h_t @ params["W_hg"].T + params["b_g"])
        c_t = f_t * c_t + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        return h_t, c_t

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.dropout_layer(self.embedding(x))      # (batch, seq, emb)
        emb = emb.permute(1, 0, 2)                       # (seq, batch, emb)
        seq_len, batch, _ = emb.size()

        h_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        c_f = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        h_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)
        c_b = torch.zeros(self.num_layers, batch, self.hidden_size, device=x.device)

        layer_input = emb
        for layer_idx in range(self.num_layers):
            fwd_params = self.fwd_layers[layer_idx]      # θ
            bwd_params = self.bwd_layers[layer_idx]      # same θ — same memory

            h_t_f = h_f[layer_idx].clone()
            c_t_f = c_f[layer_idx].clone()
            h_t_b = h_b[layer_idx].clone()
            c_t_b = c_b[layer_idx].clone()

            fwd_outputs = []
            for t in range(seq_len):
                h_t_f, c_t_f = self._lstm_cell(layer_input[t], h_t_f, c_t_f, fwd_params)
                fwd_outputs.append(h_t_f.unsqueeze(0))

            bwd_outputs = [None] * seq_len
            for t in reversed(range(seq_len)):
                h_t_b, c_t_b = self._lstm_cell(layer_input[t], h_t_b, c_t_b, bwd_params)
                bwd_outputs[t] = h_t_b.unsqueeze(0)

            fwd = torch.cat(fwd_outputs, dim=0)
            bwd = torch.cat(bwd_outputs, dim=0)
            layer_output = fwd + bwd                     # addition — preserves hidden_dim

            h_f[layer_idx] = h_t_f
            c_f[layer_idx] = c_t_f
            h_b[layer_idx] = h_t_b
            c_b[layer_idx] = c_t_b

            if layer_idx < self.num_layers - 1:
                layer_output = self.dropout_layer(layer_output)
            layer_input = layer_output

        sentence = h_f[-1] + h_b[-1]                    # final hidden from both directions
        return self.fc(self.dropout_layer(sentence))


# ============================================================
# DATA: Vocabulary + SST-2 Dataset
# ============================================================
class Vocabulary:
    def __init__(self, min_freq=2):
        self.min_freq  = min_freq
        self.word2idx  = {"<pad>": 0, "<unk>": 1}
        self.idx2word  = {0: "<pad>", 1: "<unk>"}
        self.word_freq = {}

    def build(self, sentences):
        for sent in sentences:
            for word in sent.lower().split():
                self.word_freq[word] = self.word_freq.get(word, 0) + 1
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word
        print(f"Vocabulary size: {len(self.word2idx):,}")

    def encode(self, sentence, max_len):
        tokens = sentence.lower().split()[:max_len]
        return [self.word2idx.get(t, 1) for t in tokens]

    def __len__(self):
        return len(self.word2idx)


class SST2Dataset(Dataset):
    def __init__(self, sentences, labels, vocab, max_len):
        self.data    = [(vocab.encode(s, max_len), l)
                        for s, l in zip(sentences, labels)]
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]
        padded = tokens + [0] * (self.max_len - len(tokens))
        return torch.tensor(padded, dtype=torch.long), torch.tensor(label, dtype=torch.long)


def load_sst2():
    try:
        from datasets import load_dataset
        print("Loading SST-2 from HuggingFace...")
        dataset = load_dataset("glue", "sst2")
        train_sents  = dataset["train"]["sentence"]
        train_labels = dataset["train"]["label"]
        val_sents    = dataset["validation"]["sentence"]
        val_labels   = dataset["validation"]["label"]
        print(f"Train: {len(train_sents):,}  Val: {len(val_sents):,}")
        return train_sents, train_labels, val_sents, val_labels
    except Exception as e:
        print(f"HuggingFace load failed: {e}")
        print("Falling back to manual SST-2 CSV loading...")
        # fallback if datasets library not available
        import csv
        def read_tsv(path):
            sents, labels = [], []
            with open(path) as f:
                reader = csv.DictReader(f, delimiter="\t")
                for row in reader:
                    sents.append(row["sentence"])
                    labels.append(int(row["label"]))
            return sents, labels
        train_sents, train_labels = read_tsv("/kaggle/input/sst2/train.tsv")
        val_sents,   val_labels   = read_tsv("/kaggle/input/sst2/dev.tsv")
        return train_sents, train_labels, val_sents, val_labels


def load_glove(path, vocab, emb_dim):
    print(f"Loading GloVe from {path}...")
    emb_matrix = torch.zeros(len(vocab), emb_dim)
    nn.init.normal_(emb_matrix, mean=0, std=0.1)
    emb_matrix[0] = torch.zeros(emb_dim)             # <pad> = zero vector

    found = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            word  = parts[0]
            if word in vocab.word2idx:
                emb_matrix[vocab.word2idx[word]] = torch.tensor(
                    [float(x) for x in parts[1:]], dtype=torch.float)
                found += 1

    coverage = found / (len(vocab) - 2) * 100         # exclude <pad> and <unk>
    print(f"GloVe coverage: {found:,}/{len(vocab)-2:,} = {coverage:.1f}%")
    return emb_matrix


# ============================================================
# TRAINING + EVALUATION
# ============================================================
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * y.size(0)
        correct    += (logits.argmax(1) == y).sum().item()
        total      += y.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y   = x.to(DEVICE), y.to(DEVICE)
            logits  = model(x)
            loss    = criterion(logits, y)
            total_loss += loss.item() * y.size(0)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)
    return total_loss / total, correct / total


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def run_experiment(name, model, train_loader, val_loader, cfg):
    print(f"\n{'='*55}")
    print(f"Training: {name}")
    print(f"Trainable params: {count_params(model):,}")
    print(f"{'='*55}")

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["lr"]
    )
    criterion = nn.CrossEntropyLoss()

    best_val_acc  = 0.0
    best_state    = None
    patience_cnt  = 0
    history       = []

    for epoch in range(1, cfg["n_epochs"] + 1):
        t0 = time.time()
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0

        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
        })

        print(f"Epoch {epoch:02d} | "
              f"train_loss {train_loss:.4f}  train_acc {train_acc:.4f} | "
              f"val_loss {val_loss:.4f}  val_acc {val_acc:.4f} | "
              f"{elapsed:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= cfg["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

    # restore best
    model.load_state_dict(best_state)
    print(f"\nBest val accuracy: {best_val_acc:.4f}")
    return best_val_acc, history


# ============================================================
# WEIGHT SHARING VERIFICATION
# ============================================================
def verify_sharing(model):
    print("\n── Weight Sharing Verification ──")
    same_obj  = model.fwd_layers[0] is model.bwd_layers[0]
    same_addr = (model.fwd_layers[0]["W_hi"].data_ptr() ==
                 model.bwd_layers[0]["W_hi"].data_ptr())
    print(f"  fwd_layers[0] IS bwd_layers[0]  : {same_obj}")
    print(f"  W_hi same memory address        : {same_addr}")
    return same_obj and same_addr


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":

    # ── Load SST-2 ───────────────────────────────────────────
    train_sents, train_labels, val_sents, val_labels = load_sst2()

    # ── Build vocabulary ─────────────────────────────────────
    vocab = Vocabulary(min_freq=CONFIG["min_freq"])
    vocab.build(train_sents)

    # ── Load GloVe ───────────────────────────────────────────
    glove_emb = load_glove(CONFIG["glove_path"], vocab, CONFIG["embedding_dim"])

    # ── Build datasets ───────────────────────────────────────
    train_ds = SST2Dataset(train_sents, train_labels, vocab, CONFIG["max_seq_len"])
    val_ds   = SST2Dataset(val_sents,   val_labels,   vocab, CONFIG["max_seq_len"])

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                              shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"],
                              shuffle=False, num_workers=2)

    print(f"\nTrain batches : {len(train_loader)}")
    print(f"Val batches   : {len(val_loader)}")

    # ── Build models ─────────────────────────────────────────
    model_args = (len(vocab), CONFIG["embedding_dim"], CONFIG["hidden_dim"],
                  CONFIG["n_layers"], CONFIG["dropout"], glove_emb)

    torch.manual_seed(42)
    standard_model = StandardBiLSTM(*model_args).to(DEVICE)

    torch.manual_seed(42)
    shared_model   = SharedWeightBiLSTM(*model_args).to(DEVICE)

    # ── Verify sharing before training ───────────────────────
    sharing_confirmed = verify_sharing(shared_model)

    # ── Run experiments ──────────────────────────────────────
    std_acc,    std_hist    = run_experiment(
        "Standard BiLSTM (separate weights)",
        standard_model, train_loader, val_loader, CONFIG
    )

    shared_acc, shared_hist = run_experiment(
        "Explicit SharedWeightBiLSTM",
        shared_model, train_loader, val_loader, CONFIG
    )

    # ── Save models ──────────────────────────────────────────
    torch.save(standard_model.state_dict(),
               os.path.join(CONFIG["save_dir"], "standard_bilstm.pt"))
    torch.save(shared_model.state_dict(),
               os.path.join(CONFIG["save_dir"], "shared_bilstm.pt"))

    # ── Final comparison ─────────────────────────────────────
    std_params    = count_params(standard_model)
    shared_params = count_params(shared_model)
    reduction     = (1 - shared_params / std_params) * 100
    acc_drop      = (std_acc - shared_acc) * 100

    print("\n" + "=" * 55)
    print("FINAL RESULTS")
    print("=" * 55)
    print(f"\n{'Metric':<30} {'Standard':>12} {'SharedBi':>12}")
    print("-" * 55)
    print(f"{'Best Val Accuracy':<30} {std_acc:>12.4f} {shared_acc:>12.4f}")
    print(f"{'Trainable Params':<30} {std_params:>12,} {shared_params:>12,}")
    print(f"{'Param Reduction':<30} {'baseline':>12} {reduction:>11.1f}%")
    print(f"{'Accuracy Drop':<30} {'baseline':>12} {acc_drop:>11.2f}%")
    print(f"{'Weight Sharing Confirmed':<30} {'False':>12} {str(sharing_confirmed):>12}")
    print(f"{'Output Dim':<30} {CONFIG['hidden_dim']*2:>12} {CONFIG['hidden_dim']:>12}")

    # ── Save results to JSON ─────────────────────────────────
    results = {
        "standard": {
            "best_val_acc" : std_acc,
            "params"       : std_params,
            "history"      : std_hist,
        },
        "shared": {
            "best_val_acc"        : shared_acc,
            "params"              : shared_params,
            "param_reduction_pct" : reduction,
            "acc_drop_pct"        : acc_drop,
            "sharing_confirmed"   : sharing_confirmed,
            "history"             : shared_hist,
        },
        "config": CONFIG,
    }

    results_path = os.path.join(CONFIG["save_dir"], "results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {results_path}")

Device: cuda
Loading SST-2 from HuggingFace...
Train: 67,349  Val: 872
Vocabulary size: 14,311
Loading GloVe from /kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt...
GloVe coverage: 13,318/14,309 = 93.1%

Train batches : 527
Val batches   : 7

── Weight Sharing Verification ──
  fwd_layers[0] IS bwd_layers[0]  : True
  W_hi same memory address        : True

Training: Standard BiLSTM (separate weights)
Trainable params: 235,778
Epoch 01 | train_loss 0.4761  train_acc 0.7655 | val_loss 0.4753  val_acc 0.7729 | 4.6s
Epoch 02 | train_loss 0.4087  train_acc 0.8092 | val_loss 0.4312  val_acc 0.8085 | 3.8s
Epoch 03 | train_loss 0.3821  train_acc 0.8253 | val_loss 0.3913  val_acc 0.8177 | 3.8s
Epoch 04 | train_loss 0.3558  train_acc 0.8402 | val_loss 0.4000  val_acc 0.8303 | 3.8s
Epoch 05 | train_loss 0.3383  train_acc 0.8490 | val_loss 0.4419  val_acc 0.8062 | 3.8s
Epoch 06 | train_loss 0.3208  train_acc 0.8591 | val_loss 0.3702  val_acc 0.8291 | 

| Metric | Standard BiLSTM | SharedBi LSTM |
|---|---|---|
| Accuracy | 84.17% | 83.03% |
| Parameters | 235,778 | 117,506 |
| Param Reduction | baseline | 50.2% |
| Epochs to Converge | 10 | 6 |
| Epoch 1 Val Acc | 77.29% | 80.16% ⬆ SharedBi starts better |
| Speed per Epoch | 3.8s | 70.7s ⬇ Standard wins |
| Output Dim | 256 | 128 |

## Experiments with Language Translation

In [1]:
!pip install sacrebleu datasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00


In [2]:
# ============================================================
# Bidirectional Weight Sharing for Language Translation
#
# Exp 1a: Custom LSTM seq2seq — EN→DE (separate enc/dec cells)
# Exp 1b: Custom LSTM seq2seq — DE→EN (separate enc/dec cells)
# Exp  2: One shared LSTM cell — handles both EN→DE and DE→EN
#
# Dataset : Multi30k EN↔DE
# Metric  : BLEU (sacrebleu)
# ============================================================
#
# Run these in Colab/Kaggle before executing:
#   !pip install datasets sacrebleu --quiet
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import sacrebleu
import random, time, json

# ── CONFIG ───────────────────────────────────────────────────
CONFIG = {
    'embed_dim' : 256,
    'hidden_dim': 512,
    'dropout'   : 0.3,
    'batch_size': 128,
    'lr'        : 1e-3,
    'n_epochs'  : 20,
    'patience'  : 5,
    'clip'      : 1.0,
    'tf_ratio'  : 0.5,
    'min_freq'  : 2,
    'max_len'   : 50,
    'seed'      : 42,
    'save_path' : '/kaggle/working/nmt_results.json',
    'device'    : 'cuda' if torch.cuda.is_available() else 'cpu',
}

torch.manual_seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
print(f"Device: {CONFIG['device']}")

PAD, SOS, EOS, UNK = 0, 1, 2, 3


# ── VOCABULARY ───────────────────────────────────────────────
class Vocab:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.w2i = {'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3}
        self.i2w = {0: '<pad>', 1: '<sos>', 2: '<eos>', 3: '<unk>'}

    def build(self, sentences):
        cnt = Counter(w for s in sentences for w in s.lower().split())
        for w, f in cnt.items():
            if f >= self.min_freq and w not in self.w2i:
                i = len(self.w2i)
                self.w2i[w] = i
                self.i2w[i] = w

    def encode(self, s, max_len=None):
        toks = s.lower().split()
        if max_len:
            toks = toks[:max_len]
        return [SOS] + [self.w2i.get(t, UNK) for t in toks] + [EOS]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS:
                break
            if i not in (PAD, SOS):
                out.append(self.i2w.get(i, '<unk>'))
        return ' '.join(out)

    def __len__(self):
        return len(self.w2i)


# ── DATA LOADING ─────────────────────────────────────────────
def load_data():
    print("\nLoading Multi30k from HuggingFace...")
    try:
        from datasets import load_dataset
        ds = load_dataset("bentrevett/multi30k")
        tr_en = [x['en'] for x in ds['train']]
        tr_de = [x['de'] for x in ds['train']]
        va_en = [x['en'] for x in ds['validation']]
        va_de = [x['de'] for x in ds['validation']]
        te_en = [x['en'] for x in ds['test']]
        te_de = [x['de'] for x in ds['test']]
    except Exception as e:
        raise RuntimeError(f"Could not load Multi30k: {e}\n"
                           "Run: !pip install datasets")

    print(f"Train {len(tr_en)} | Val {len(va_en)} | Test {len(te_en)}")

    en_vocab = Vocab(CONFIG['min_freq'])
    de_vocab = Vocab(CONFIG['min_freq'])
    en_vocab.build(tr_en)
    de_vocab.build(tr_de)
    print(f"EN vocab {len(en_vocab):,} | DE vocab {len(de_vocab):,}")

    return tr_en, tr_de, va_en, va_de, te_en, te_de, en_vocab, de_vocab


class NMTDataset(Dataset):
    def __init__(self, src, tgt, sv, tv, ml=50):
        self.data = [
            (torch.tensor(sv.encode(s, ml)),
             torch.tensor(tv.encode(t, ml)))
            for s, t in zip(src, tgt)
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]


def collate(batch):
    s, t = zip(*batch)
    return (pad_sequence(s, batch_first=True, padding_value=PAD),
            pad_sequence(t, batch_first=True, padding_value=PAD))


def make_loaders(tr_en, tr_de, va_en, va_de, te_en, te_de,
                 en_vocab, de_vocab):
    ML, BS = CONFIG['max_len'], CONFIG['batch_size']

    def dl(src, tgt, sv, tv, shuffle):
        return DataLoader(NMTDataset(src, tgt, sv, tv, ML),
                          BS, shuffle=shuffle, collate_fn=collate)

    en2de = {
        'train': dl(tr_en, tr_de, en_vocab, de_vocab, True),
        'val':   dl(va_en, va_de, en_vocab, de_vocab, False),
        'test':  dl(te_en, te_de, en_vocab, de_vocab, False),
    }
    de2en = {
        'train': dl(tr_de, tr_en, de_vocab, en_vocab, True),
        'val':   dl(va_de, va_en, de_vocab, en_vocab, False),
        'test':  dl(te_de, te_en, de_vocab, en_vocab, False),
    }
    print(f"EN→DE train batches: {len(en2de['train'])} | "
          f"DE→EN train batches: {len(de2en['train'])}")
    return en2de, de2en


# ── LSTM CELL (Adigun-style ParameterDict) ───────────────────
class LSTMCell(nn.Module):
    """
    Custom LSTM cell with explicit ParameterDict.
    Direction-blind: the cell has no notion of forward vs backward,
    source vs target, or English vs German. Those roles live in the
    pipeline that calls it.
    """
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        s = 0.01
        self.p = nn.ParameterDict({
            'Wii': nn.Parameter(torch.randn(hidden_size, input_size)  * s),
            'Whi': nn.Parameter(torch.randn(hidden_size, hidden_size) * s),
            'bi' : nn.Parameter(torch.zeros(hidden_size)),
            'Wif': nn.Parameter(torch.randn(hidden_size, input_size)  * s),
            'Whf': nn.Parameter(torch.randn(hidden_size, hidden_size) * s),
            'bf' : nn.Parameter(torch.zeros(hidden_size)),
            'Wig': nn.Parameter(torch.randn(hidden_size, input_size)  * s),
            'Whg': nn.Parameter(torch.randn(hidden_size, hidden_size) * s),
            'bg' : nn.Parameter(torch.zeros(hidden_size)),
            'Wio': nn.Parameter(torch.randn(hidden_size, input_size)  * s),
            'Who': nn.Parameter(torch.randn(hidden_size, hidden_size) * s),
            'bo' : nn.Parameter(torch.zeros(hidden_size)),
        })

    def forward(self, x, h, c):
        p = self.p
        i = torch.sigmoid(x @ p['Wii'].T + h @ p['Whi'].T + p['bi'])
        f = torch.sigmoid(x @ p['Wif'].T + h @ p['Whf'].T + p['bf'])
        g = torch.tanh(   x @ p['Wig'].T + h @ p['Whg'].T + p['bg'])
        o = torch.sigmoid(x @ p['Wio'].T + h @ p['Who'].T + p['bo'])
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new


# ── EXP 1: STANDARD SEQ2SEQ (separate encoder + decoder cells)
class Exp1Seq2Seq(nn.Module):
    """
    Standard seq2seq with two separate LSTM cells:
      enc_cell — encodes source sequence
      dec_cell — decodes into target tokens
    enc_cell IS NOT dec_cell.
    One model handles one translation direction only.
    """
    def __init__(self, src_vsz, tgt_vsz, emb_dim, hid_dim, drop=0.3):
        super().__init__()
        self.emb_src  = nn.Embedding(src_vsz, emb_dim, padding_idx=PAD)
        self.emb_tgt  = nn.Embedding(tgt_vsz, emb_dim, padding_idx=PAD)
        self.enc_cell = LSTMCell(emb_dim, hid_dim)   # encoder weights
        self.dec_cell = LSTMCell(emb_dim, hid_dim)   # decoder weights — SEPARATE
        self.fc       = nn.Linear(hid_dim, tgt_vsz)
        self.drop     = nn.Dropout(drop)
        self.hid_dim  = hid_dim

    def _encode(self, src):
        B, dev = src.size(0), src.device
        x = self.drop(self.emb_src(src))
        h = torch.zeros(B, self.hid_dim, device=dev)
        c = torch.zeros(B, self.hid_dim, device=dev)
        for t in range(x.size(1)):
            h, c = self.enc_cell(x[:, t], h, c)
        return h, c

    def forward(self, src, tgt, tf_ratio=0.5):
        B, T = tgt.shape
        V    = self.fc.out_features
        dev  = src.device
        h, c = self._encode(src)
        out  = torch.zeros(B, T, V, device=dev)
        tok  = tgt[:, 0]
        for t in range(1, T):
            e = self.drop(self.emb_tgt(tok))
            h, c = self.dec_cell(e, h, c)
            pred = self.fc(h)
            out[:, t] = pred
            tok = tgt[:, t] if random.random() < tf_ratio else pred.argmax(1)
        return out

    def translate(self, src, max_len=50):
        self.eval()
        dev = src.device
        with torch.no_grad():
            h, c = self._encode(src)
            tok  = torch.full((src.size(0),), SOS, device=dev)
            preds = []
            for _ in range(max_len):
                e = self.emb_tgt(tok)
                h, c = self.dec_cell(e, h, c)
                tok = self.fc(h).argmax(1)
                preds.append(tok)
                if (tok == EOS).all():
                    break
        return torch.stack(preds, dim=1)


# ── EXP 2: SHARED CELL SEQ2SEQ ───────────────────────────────
class Exp2SharedSeq2Seq(nn.Module):
    """
    One LSTMCell shared across ALL four roles:
      1. Encoder for EN→DE
      2. Decoder for EN→DE
      3. Encoder for DE→EN
      4. Decoder for DE→EN

    The cell is direction-blind, role-blind, and language-blind.
    Direction and role are determined by the pipeline.
    Language is handled by separate embedding matrices.

    LSTM params: 1x  (vs 4x for two standard Exp1 models combined)
    """
    def __init__(self, en_vsz, de_vsz, emb_dim, hid_dim, drop=0.3):
        super().__init__()
        # Language-specific embeddings (language identity lives here)
        self.emb_en = nn.Embedding(en_vsz, emb_dim, padding_idx=PAD)
        self.emb_de = nn.Embedding(de_vsz, emb_dim, padding_idx=PAD)

        # ── THE SHARED CELL ──────────────────────────────────
        # Same object. Same memory address. All four roles above
        # call self.cell — there is no other LSTM cell in this model.
        self.cell   = LSTMCell(emb_dim, hid_dim)
        # ─────────────────────────────────────────────────────

        # Language-specific output projections
        self.fc_de  = nn.Linear(hid_dim, de_vsz)  # predicts German tokens
        self.fc_en  = nn.Linear(hid_dim, en_vsz)  # predicts English tokens

        self.drop   = nn.Dropout(drop)
        self.hid_dim = hid_dim

    def _encode(self, emb, dev):
        """Run embedded sequence through the shared cell."""
        B = emb.size(0)
        h = torch.zeros(B, self.hid_dim, device=dev)
        c = torch.zeros(B, self.hid_dim, device=dev)
        for t in range(emb.size(1)):
            h, c = self.cell(emb[:, t], h, c)   # shared cell — encoder role
        return h, c

    def forward(self, src, tgt, direction='en2de', tf_ratio=0.5):
        dev = src.device
        B, T = tgt.shape

        # Select language-specific components based on direction
        if direction == 'en2de':
            src_emb    = self.drop(self.emb_en(src))
            tgt_emb_fn = lambda tok: self.drop(self.emb_de(tok))
            fc         = self.fc_de
        else:  # de2en
            src_emb    = self.drop(self.emb_de(src))
            tgt_emb_fn = lambda tok: self.drop(self.emb_en(tok))
            fc         = self.fc_en

        h, c = self._encode(src_emb, dev)

        V   = fc.out_features
        out = torch.zeros(B, T, V, device=dev)
        tok = tgt[:, 0]

        for t in range(1, T):
            e = tgt_emb_fn(tok)
            h, c = self.cell(e, h, c)           # shared cell — decoder role
            pred = fc(h)
            out[:, t] = pred
            tok = tgt[:, t] if random.random() < tf_ratio else pred.argmax(1)

        return out

    def translate(self, src, direction='en2de', max_len=50):
        self.eval()
        dev = src.device
        with torch.no_grad():
            if direction == 'en2de':
                src_emb    = self.emb_en(src)
                tgt_emb_fn = self.emb_de
                fc         = self.fc_de
            else:
                src_emb    = self.emb_de(src)
                tgt_emb_fn = self.emb_en
                fc         = self.fc_en

            h, c  = self._encode(src_emb, dev)
            tok   = torch.full((src.size(0),), SOS, device=dev)
            preds = []

            for _ in range(max_len):
                e = tgt_emb_fn(tok)
                h, c = self.cell(e, h, c)
                tok = fc(h).argmax(1)
                preds.append(tok)
                if (tok == EOS).all():
                    break

        return torch.stack(preds, dim=1)


# ── TRAINING ─────────────────────────────────────────────────
def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


criterion = nn.CrossEntropyLoss(ignore_index=PAD)


def train_exp1(model, tr_dl, va_dl, name):
    """Training loop for a single-direction Exp1 model."""
    dev  = CONFIG['device']
    opt  = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    best_val = 1e9; patience = 0; best_st = None

    print(f"\n{'='*55}")
    print(f"Training : {name}")
    print(f"Params   : {count_params(model):,}")
    print(f"{'='*55}")

    for ep in range(1, CONFIG['n_epochs'] + 1):
        t0 = time.time()

        # train
        model.train(); tr = 0.0
        for src, tgt in tr_dl:
            src, tgt = src.to(dev), tgt.to(dev)
            opt.zero_grad()
            out  = model(src, tgt, tf_ratio=CONFIG['tf_ratio'])
            loss = criterion(out[:, 1:].reshape(-1, out.size(-1)),
                             tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG['clip'])
            opt.step()
            tr += loss.item()
        tr /= len(tr_dl)

        # val
        model.eval(); va = 0.0
        with torch.no_grad():
            for src, tgt in va_dl:
                src, tgt = src.to(dev), tgt.to(dev)
                out  = model(src, tgt, tf_ratio=0.0)
                va  += criterion(out[:, 1:].reshape(-1, out.size(-1)),
                                 tgt[:, 1:].reshape(-1)).item()
        va /= len(va_dl)
        sched.step(va)

        print(f"Epoch {ep:02d} | train {tr:.4f} | val {va:.4f} | {time.time()-t0:.1f}s")

        if va < best_val:
            best_val = va; patience = 0
            best_st = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= CONFIG['patience']:
                print(f"Early stop at epoch {ep}")
                break

    model.load_state_dict(best_st)
    return model


def train_exp2(model, en2de, de2en, name):
    """
    Training loop for shared model.
    Alternates EN→DE and DE→EN batches each iteration.
    Gradient from both directions accumulates into the same cell.
    """
    dev   = CONFIG['device']
    opt   = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    best_val = 1e9; patience = 0; best_st = None

    print(f"\n{'='*55}")
    print(f"Training : {name}")
    print(f"Params   : {count_params(model):,}")
    print(f"{'='*55}")

    for ep in range(1, CONFIG['n_epochs'] + 1):
        t0 = time.time()

        # train — alternate directions
        model.train(); tr = 0.0; n = 0
        for (se, td), (sd, te) in zip(en2de['train'], de2en['train']):
            for src, tgt, dirn in [(se, td, 'en2de'), (sd, te, 'de2en')]:
                src, tgt = src.to(dev), tgt.to(dev)
                opt.zero_grad()
                out  = model(src, tgt, direction=dirn, tf_ratio=CONFIG['tf_ratio'])
                loss = criterion(out[:, 1:].reshape(-1, out.size(-1)),
                                 tgt[:, 1:].reshape(-1))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG['clip'])
                opt.step()
                tr += loss.item(); n += 1
        tr /= n

        # val — both directions
        model.eval(); va = 0.0; n = 0
        with torch.no_grad():
            for (se, td), (sd, te) in zip(en2de['val'], de2en['val']):
                for src, tgt, dirn in [(se, td, 'en2de'), (sd, te, 'de2en')]:
                    src, tgt = src.to(dev), tgt.to(dev)
                    out  = model(src, tgt, direction=dirn, tf_ratio=0.0)
                    va  += criterion(out[:, 1:].reshape(-1, out.size(-1)),
                                     tgt[:, 1:].reshape(-1)).item()
                    n   += 1
        va /= n
        sched.step(va)

        print(f"Epoch {ep:02d} | train {tr:.4f} | val {va:.4f} | {time.time()-t0:.1f}s")

        if va < best_val:
            best_val = va; patience = 0
            best_st = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= CONFIG['patience']:
                print(f"Early stop at epoch {ep}")
                break

    model.load_state_dict(best_st)
    return model


# ── EVALUATION ───────────────────────────────────────────────
def compute_bleu(model, loader, tgt_vocab, dev, direction=None, max_len=50):
    model.eval()
    hyps, refs = [], []
    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(dev)
            if direction:
                trans = model.translate(src, direction=direction, max_len=max_len)
            else:
                trans = model.translate(src, max_len=max_len)
            for i in range(src.size(0)):
                hyps.append(tgt_vocab.decode(trans[i].tolist()))
                refs.append(tgt_vocab.decode(tgt[i].tolist()))
    return sacrebleu.corpus_bleu(hyps, [refs]).score


def show_translations(model, src_sents, src_vocab, tgt_vocab, dev,
                      direction=None, n=5, label=""):
    """Print a few example translations for qualitative inspection."""
    ML = CONFIG['max_len']
    print(f"\n── Sample Translations: {label} ──")
    model.eval()
    with torch.no_grad():
        for sent in src_sents[:n]:
            ids = torch.tensor(src_vocab.encode(sent, ML)).unsqueeze(0).to(dev)
            if direction:
                out = model.translate(ids, direction=direction)
            else:
                out = model.translate(ids)
            print(f"  SRC : {sent}")
            print(f"  TGT : {tgt_vocab.decode(out[0].tolist())}")
            print()


# ── WEIGHT SHARING VERIFICATION ─────────────────────────────
def verify_sharing():
    print("\n── Weight Sharing Verification ──")

    # Exp 1: enc and dec must be different objects
    m1 = Exp1Seq2Seq(10, 10, 32, 64)
    assert m1.enc_cell is not m1.dec_cell, "Exp1 sharing check failed"
    print(f"  Exp1 enc_cell IS dec_cell          : {m1.enc_cell is m1.dec_cell}  ✓ (should be False)")

    # Exp 2: one cell used in all four paths
    m2 = Exp2SharedSeq2Seq(10, 10, 32, 64)
    # Confirm there is exactly ONE LSTMCell in the model
    cells = [m for m in m2.modules() if isinstance(m, LSTMCell)]
    assert len(cells) == 1, f"Expected 1 LSTMCell, found {len(cells)}"
    print(f"  Exp2 LSTMCell count in model       : {len(cells)}  ✓ (should be 1)")
    print(f"  Exp2 cell id (encoder = decoder)   : {id(m2.cell)}  ✓ (same object all paths)")

    del m1, m2


# ── MAIN ─────────────────────────────────────────────────────
if __name__ == '__main__':

    # Data
    (tr_en, tr_de, va_en, va_de,
     te_en, te_de, en_vocab, de_vocab) = load_data()
    en2de, de2en = make_loaders(tr_en, tr_de, va_en, va_de,
                                te_en, te_de, en_vocab, de_vocab)

    # Verify
    verify_sharing()

    # Dimensions
    dev = CONFIG['device']
    E   = CONFIG['embed_dim']
    H   = CONFIG['hidden_dim']
    D   = CONFIG['dropout']

    # ── BUILD ────────────────────────────────────────────────
    model_en2de  = Exp1Seq2Seq(len(en_vocab), len(de_vocab), E, H, D).to(dev)
    model_de2en  = Exp1Seq2Seq(len(de_vocab), len(en_vocab), E, H, D).to(dev)
    model_shared = Exp2SharedSeq2Seq(len(en_vocab), len(de_vocab), E, H, D).to(dev)

    # ── EXP 1a: EN→DE ────────────────────────────────────────
    model_en2de = train_exp1(model_en2de,
                             en2de['train'], en2de['val'],
                             "Exp1a: EN→DE  (separate enc/dec cells)")

    # ── EXP 1b: DE→EN ────────────────────────────────────────
    model_de2en = train_exp1(model_de2en,
                             de2en['train'], de2en['val'],
                             "Exp1b: DE→EN  (separate enc/dec cells)")

    # ── EXP 2: SHARED ────────────────────────────────────────
    model_shared = train_exp2(model_shared, en2de, de2en,
                              "Exp2:  EN↔DE  (one shared LSTM cell)")

    # ── BLEU ─────────────────────────────────────────────────
    print(f"\n{'='*55}\nBLEU Evaluation (Test Set)\n{'='*55}")

    b1_en2de = compute_bleu(model_en2de,  en2de['test'], de_vocab, dev)
    b1_de2en = compute_bleu(model_de2en,  de2en['test'], en_vocab, dev)
    b2_en2de = compute_bleu(model_shared, en2de['test'], de_vocab, dev, direction='en2de')
    b2_de2en = compute_bleu(model_shared, de2en['test'], en_vocab, dev, direction='de2en')

    p1 = count_params(model_en2de) + count_params(model_de2en)
    p2 = count_params(model_shared)

    # LSTM cell params only
    lstm_p1 = (sum(p.numel() for p in model_en2de.enc_cell.parameters()) +
               sum(p.numel() for p in model_en2de.dec_cell.parameters()) +
               sum(p.numel() for p in model_de2en.enc_cell.parameters()) +
               sum(p.numel() for p in model_de2en.dec_cell.parameters()))
    lstm_p2 = sum(p.numel() for p in model_shared.cell.parameters())

    print(f"\n{'Metric':<38} {'Exp1 (Separate)':>16} {'Exp2 (Shared)':>14}")
    print("-" * 70)
    print(f"{'EN→DE BLEU':<38} {b1_en2de:>16.2f} {b2_en2de:>14.2f}")
    print(f"{'DE→EN BLEU':<38} {b1_de2en:>16.2f} {b2_de2en:>14.2f}")
    print(f"{'Avg BLEU':<38} {(b1_en2de+b1_de2en)/2:>16.2f} {(b2_en2de+b2_de2en)/2:>14.2f}")
    print(f"{'Total params (both directions)':<38} {p1:>16,} {p2:>14,}")
    print(f"{'Total param reduction':<38} {'baseline':>16} {(1-p2/p1)*100:>13.1f}%")
    print(f"{'LSTM cell params only':<38} {lstm_p1:>16,} {lstm_p2:>14,}")
    print(f"{'LSTM cell reduction':<38} {'baseline':>16} {(1-lstm_p2/lstm_p1)*100:>13.1f}%")
    print(f"{'LSTM cell count':<38} {'4 cells':>16} {'1 cell':>14}")
    print(f"{'Weight sharing confirmed':<38} {'False':>16} {'True':>14}")

    # ── SAMPLE TRANSLATIONS ──────────────────────────────────
    samples = te_en[:5]
    show_translations(model_en2de, samples, en_vocab, de_vocab, dev,
                      label="Exp1a EN→DE")
    show_translations(model_shared, samples, en_vocab, de_vocab, dev,
                      direction='en2de', label="Exp2 EN→DE (shared cell)")

    de_samples = te_de[:5]
    show_translations(model_de2en, de_samples, de_vocab, en_vocab, dev,
                      label="Exp1b DE→EN")
    show_translations(model_shared, de_samples, de_vocab, en_vocab, dev,
                      direction='de2en', label="Exp2 DE→EN (shared cell)")

    # ── SAVE ─────────────────────────────────────────────────
    results = {
        'exp1_en2de_bleu': round(b1_en2de, 4),
        'exp1_de2en_bleu': round(b1_de2en, 4),
        'exp1_avg_bleu':   round((b1_en2de + b1_de2en) / 2, 4),
        'exp2_en2de_bleu': round(b2_en2de, 4),
        'exp2_de2en_bleu': round(b2_de2en, 4),
        'exp2_avg_bleu':   round((b2_en2de + b2_de2en) / 2, 4),
        'params_exp1_combined': p1,
        'params_exp2':          p2,
        'total_reduction_pct':  round((1 - p2 / p1) * 100, 2),
        'lstm_params_exp1':     lstm_p1,
        'lstm_params_exp2':     lstm_p2,
        'lstm_reduction_pct':   round((1 - lstm_p2 / lstm_p1) * 100, 2),
        'lstm_cells_exp1':      4,
        'lstm_cells_exp2':      1,
    }
    with open(CONFIG['save_path'], 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {CONFIG['save_path']}")

Device: cuda

Loading Multi30k from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train 29000 | Val 1014 | Test 1000
EN vocab 7,704 | DE vocab 9,597
EN→DE train batches: 227 | DE→EN train batches: 227

── Weight Sharing Verification ──
  Exp1 enc_cell IS dec_cell          : False  ✓ (should be False)
  Exp2 LSTMCell count in model       : 1  ✓ (should be 1)
  Exp2 cell id (encoder = decoder)   : 138469610840976  ✓ (same object all paths)

Training : Exp1a: EN→DE  (separate enc/dec cells)
Params   : 12,502,141
Epoch 01 | train 5.4041 | val 5.2314 | 30.3s
Epoch 02 | train 4.6653 | val 4.8430 | 29.9s
Epoch 03 | train 4.2150 | val 4.4869 | 30.4s
Epoch 04 | train 3.8816 | val 4.3370 | 30.8s
Epoch 05 | train 3.6572 | val 4.1332 | 31.1s
Epoch 06 | train 3.4476 | val 4.0194 | 30.7s
Epoch 07 | train 3.2628 | val 3.9254 | 30.9s
Epoch 08 | train 3.0885 | val 3.8614 | 31.0s
Epoch 09 | train 2.9573 | val 3.8252 | 30.9s
Epoch 10 | train 2.8198 | val 3.7986 | 30.9s
Epoch 11 | train 2.7037 | val 3.7625 | 31.0s
Epoch 12 | train 2.5949 | val 3.7091 | 31.1s
Epoch 13 | train 2.4773 | v